# Non-Standard Dataset Extraction

Extracts a sample of ~40k loans per year (10k per quarter) from the Freddie Mac
Non-Standard Dataset (ARM, interest-only, limited documentation loans).

These are the loan types excluded from the Standard Dataset for not meeting
Credit Risk Transfer eligibility criteria — and are believed to be the riskier
loans that drove the 2008 crisis.

Output: a single Parquet file with origination data + default target,
ready to be merged with the Standard Dataset.

In [ ]:
import zipfile
import io
import pandas as pd
from pathlib import Path

In [ ]:
# Configuration
NSD_FOLDER = Path('../data/01_raw/non_standard')  # adjust to your actual path
YEARS = list(range(2000, 2009))
QUARTERS = ['Q1', 'Q2', 'Q3', 'Q4']
SAMPLE_PER_QUARTER = 10000  # 10k per quarter -> ~40k per year
RANDOM_STATE = 42

# NSD origination files have 31 columns (missing mi_cancellation_indicator
# compared to the Standard Dataset's 32 columns)
ORIGINATION_COLUMNS_NSD = [
    'credit_score',
    'first_payment_date',
    'first_time_homebuyer_flag',
    'maturity_date',
    'msa',
    'mi_percentage',
    'number_of_units',
    'occupancy_status',
    'original_cltv',
    'original_dti',
    'original_upb',
    'original_ltv',
    'original_interest_rate',
    'channel',
    'prepayment_penalty_flag',
    'amortization_type',
    'property_state',
    'property_type',
    'postal_code',
    'loan_sequence_number',
    'loan_purpose',
    'original_loan_term',
    'number_of_borrowers',
    'seller_name',
    'servicer_name',
    'super_conforming_flag',
    'pre_relief_refinance_loan_sequence_number',
    'special_eligibility_program',
    'relief_refinance_indicator',
    'property_valuation_method',
    'interest_only_indicator',
]

DEFAULT_CODES = ['02', '03', '09']

## Helper functions

Files are nested: `historical_data_excl_{year}.zip` contains
`historical_data_excl_{year}{quarter}.zip`, which in turn contains the
actual `.txt` data files. We read everything in-memory without extracting
to disk.

In [ ]:
def read_nsd_quarter_origination(year: int, quarter: str, sample_size: int) -> pd.DataFrame:
    """Read and sample origination data for a single NSD quarter.

    Args:
        year: origination year.
        quarter: quarter string, e.g. 'Q1'.
        sample_size: number of loans to randomly sample.
    Returns:
        Sampled origination DataFrame with named columns, or empty
        DataFrame if the file is missing or unreadable.
    """
    outer_zip_path = NSD_FOLDER / f'historical_data_excl_{year}.zip'

    if not outer_zip_path.exists():
        print(f'  [skip] {year}{quarter}: outer zip not found')
        return pd.DataFrame()

    try:
        with zipfile.ZipFile(outer_zip_path) as outer_zip:
            inner_zip_name = f'historical_data_excl_{year}{quarter}.zip'

            with outer_zip.open(inner_zip_name) as inner_zip_bytes:
                inner_zip_data = io.BytesIO(inner_zip_bytes.read())

                with zipfile.ZipFile(inner_zip_data) as inner_zip:
                    txt_name = f'historical_data_excl_{year}{quarter}.txt'

                    with inner_zip.open(txt_name) as f:
                        df = pd.read_csv(
                            f,
                            sep='|',
                            header=None,
                            names=ORIGINATION_COLUMNS_NSD,
                            low_memory=False,
                        )

        if len(df) > sample_size:
            df = df.sample(n=sample_size, random_state=RANDOM_STATE)

        print(f'  [ok]   {year}{quarter}: {len(df):,} loans sampled')
        return df

    except KeyError:
        print(f'  [skip] {year}{quarter}: inner file not found in archive')
        return pd.DataFrame()
    except Exception as e:
        print(f'  [error] {year}{quarter}: {e}')
        return pd.DataFrame()

In [ ]:
def read_nsd_quarter_performance(year: int, quarter: str, loan_ids: set) -> pd.DataFrame:
    """Read performance data for a single NSD quarter, filtered to the
    sampled loan_sequence_numbers only (to keep memory usage low).

    The performance file's zero_balance_code is in column index 8
    (0-indexed), matching the Standard Dataset's PERFORMANCE_COLUMNS layout.

    Args:
        year: origination year.
        quarter: quarter string, e.g. 'Q1'.
        loan_ids: set of loan_sequence_number values to keep.
    Returns:
        DataFrame with loan_sequence_number and zero_balance_code only.
    """
    outer_zip_path = NSD_FOLDER / f'historical_data_excl_{year}.zip'

    if not outer_zip_path.exists():
        return pd.DataFrame()

    try:
        with zipfile.ZipFile(outer_zip_path) as outer_zip:
            inner_zip_name = f'historical_data_excl_{year}{quarter}.zip'

            with outer_zip.open(inner_zip_name) as inner_zip_bytes:
                inner_zip_data = io.BytesIO(inner_zip_bytes.read())

                with zipfile.ZipFile(inner_zip_data) as inner_zip:
                    txt_name = f'historical_data_excl_time_{year}{quarter}.txt'

                    with inner_zip.open(txt_name) as f:
                        # Only load columns 0 (loan_sequence_number) and 8 (zero_balance_code)
                        df = pd.read_csv(
                            f,
                            sep='|',
                            header=None,
                            usecols=[0, 8],
                            names=['loan_sequence_number', 'zero_balance_code'],
                            low_memory=False,
                        )

        df = df[df['loan_sequence_number'].isin(loan_ids)]
        return df

    except KeyError:
        return pd.DataFrame()
    except Exception as e:
        print(f'  [error] performance {year}{quarter}: {e}')
        return pd.DataFrame()

## Extract origination samples for all years

In [ ]:
origination_frames = []

for year in YEARS:
    print(f'Year {year}:')
    for quarter in QUARTERS:
        df_q = read_nsd_quarter_origination(year, quarter, SAMPLE_PER_QUARTER)
        if not df_q.empty:
            df_q['year'] = year
            origination_frames.append(df_q)

nsd_origination = pd.concat(origination_frames, ignore_index=True)
print(f'\nTotal NSD origination sample: {len(nsd_origination):,} loans')
nsd_origination.head()

## Extract performance data for the sampled loans

We only fetch performance records for the loan_sequence_numbers we already
sampled, to avoid loading the full multi-GB performance files.

In [ ]:
sampled_loan_ids = set(nsd_origination['loan_sequence_number'])

performance_frames = []

for year in YEARS:
    print(f'Performance — Year {year}:')
    for quarter in QUARTERS:
        df_perf = read_nsd_quarter_performance(year, quarter, sampled_loan_ids)
        if not df_perf.empty:
            performance_frames.append(df_perf)
            print(f'  [ok]   {year}{quarter}: {len(df_perf):,} performance records')

nsd_performance = pd.concat(performance_frames, ignore_index=True)
print(f'\nTotal NSD performance records: {len(nsd_performance):,}')

## Build the default target

A loan is considered defaulted if its zero_balance_code is ever 02, 03, or 09.

In [ ]:
nsd_performance['zero_balance_code_clean'] = (
    pd.to_numeric(nsd_performance['zero_balance_code'], errors='coerce')
    .astype('Int64')
    .astype('string')
    .str.zfill(2)
)

nsd_target = (
    nsd_performance.groupby('loan_sequence_number')['zero_balance_code_clean']
    .apply(lambda codes: int(codes.isin(DEFAULT_CODES).any()))
    .reset_index()
    .rename(columns={'zero_balance_code_clean': 'default'})
)

print(f'Loans with target: {len(nsd_target):,}')
print(f'Default rate: {nsd_target["default"].mean()*100:.2f}%')

## Join origination with target

In [ ]:
nsd_full = nsd_origination.merge(nsd_target, on='loan_sequence_number', how='inner')
nsd_full['default'] = nsd_full['default'].astype(int)
nsd_full['source'] = 'non_standard'

print(f'Final NSD dataset: {len(nsd_full):,} loans')
print(f'\nDefault rate by year:')
print(nsd_full.groupby('year')['default'].agg(['count', 'mean']))

## Quick comparison: ARM vs FRM default rates

In [ ]:
print('Amortization type distribution:')
print(nsd_full['amortization_type'].value_counts())
print()
print('Default rate by amortization type:')
print(nsd_full.groupby('amortization_type')['default'].mean())

## Save to Parquet for later use

In [ ]:
output_path = Path('../data/02_intermediate/nsd_sample.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)
nsd_full.to_parquet(output_path, index=False)
print(f'Saved to {output_path}')
print(f'Shape: {nsd_full.shape}')